# Pista A — evidencia observacional de la adopción de productos (A1–A4)

**19 de septiembre de 2026.** Corre la pista A de `docs/experimentos_productos.md` §4 sobre el
dataset real: cuándo adopta cada empresa un producto de financiación (A1), qué le pasa después
(A2), quién lo adopta (A3) y si el efecto depende del colchón de caja (A4).

Este cuaderno **importa `xray`, no define el pipeline**: los eventos salen de `xray.adoption`,
los estimadores de `xray.eventstudy`, la propensión de `xray.behavior` y el panel puntuado de
`xray.rules.run` sobre `artifacts/features.parquet`. Aquí solo se orquesta, se dibuja y se guarda.

Salidas en `artifacts/experiments/`: `A1_events.csv`, `A1_counts.csv`, `A2_att.csv`,
`A2_att.png`, `A3_propensity.csv`, `A3_overlap.png`, `A4_uplift.csv`, `A4_uplift.png` y
`A_summary.json`.

**Advertencia que vale para todo el cuaderno:** el dataset es sintético y no hay registro de
ofertas (§2.1). Nada de lo que sigue es un efecto causal de una recomendación; son diferencias
medias entre quien adoptó y quien no, con su pre-tendencia al lado como contraste de falsación.

In [ ]:
import json
import time
import warnings

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy.stats import spearmanr

from xray import adoption, behavior, eventstudy, rules
from xray import features as features_mod
from xray.data import artifacts_dir, load

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

T0 = time.time()
OUT = artifacts_dir() / "experiments"
OUT.mkdir(parents=True, exist_ok=True)

# Combinaciones (producto, fuente) del estudio de eventos, en orden de interés.
A2_PAIRS = [
    ("loan", "installment"),
    ("loan", "step_up"),
    ("line", "line_tx"),
    ("factoring", "description"),
    ("anticipo", "description"),
    ("disposicion", "description"),
    ("loan", "created_at"),
    ("line", "created_at"),
]
MIN_CLEAN_EVENTS = 8       # por debajo de esto no se estima nada
HORIZONS = (1, 3, 6)
A3_PAIRS = [("loan", "installment"), ("line", "line_tx"), ("factoring", "description")]
A3_HORIZON = 3
A4_SPLIT = "2025-08"       # eventos hasta aquí entrenan; los posteriores son test
A4_PARAMS = dict(n_estimators=200, num_leaves=15, min_child_samples=30, verbose=-1)
A4_CONTROLS = 5000
SEED = 0
print(f"salidas en {OUT}")

## Carga

`load()` trae las tablas del reto desde la caché parquet y `features.parquet` es la tabla del
seam `features(company_id, month)`. `rules.run` añade rangos, índice de estado, nivel, score,
outlook y `n_red`.

`features_mod.derive` solo se llama **si faltan** las señales v2, y se usa lo que ya está en el
parquet cuando están. No es cosmético: recalcular `net_cash_flow_ratio_3m` con `derive` cambia
el 0,28 % de las filas (máximo |diff| 5,9) respecto a lo guardado, así que recalcular por
costumbre movería los números de este cuaderno sin que nadie lo hubiera pedido.

In [ ]:
tables = load()
companies = tables["companies"]
features = pd.read_parquet(artifacts_dir() / "features.parquet")
if "net_cash_flow_ratio_3m" not in features.columns:
    features = features_mod.derive(features)
scored = rules.run(features)
groups = companies.set_index("company_id")["group_id"]

# Panel del estudio de eventos: el panel puntuado más los dos resultados binarios.
panel = eventstudy.future_any_below(scored, "min_balance_eur", 6, 0.0, "breach6")
panel["red"] = (panel["n_red"] >= 2).astype(float)
panel["mo"] = pd.PeriodIndex(panel["month"].astype(str), freq="M").asi8  # mes como entero

print(f"features {features.shape} · panel puntuado {panel.shape}")
print(f"meses {panel['month'].min()} … {panel['month'].max()} · {panel['company_id'].nunique()} empresas")
print(f"breach6: media {panel['breach6'].mean():.3f}, NaN {panel['breach6'].isna().mean():.1%} "
      f"(los últimos 6 meses de cada empresa no tienen futuro observable)")
print(f"mes rojo (n_red ≥ 2): {panel['red'].mean():.3f}")

## A1 — Tabla de adopción

`adoption.adoption_events` fecha la adopción con precisión de mes desde los movimientos (primera
cuota, salto de cuota, primer movimiento en cuenta de línea, abonos con FACTORING / ANTICIPO /
DISPOSICION / CONFIRMING) y añade `created_at` como fuente secundaria. `clean_events` se queda
con el primer evento por (empresa, producto, fuente) que tenga ≥ 6 meses de historia antes y
≥ 6 meses después: son los únicos que sirven para medir un efecto.

In [ ]:
events = adoption.adoption_events(
    tables["transactions"], tables["debt_products"], tables["banking_products"], features
)
clean_all = adoption.clean_events(events)
key = ["company_id", "month", "product", "source"]
events["clean"] = events[key].merge(clean_all[key].assign(k=1), how="left", on=key)["k"].notna().to_numpy()

counts = (
    events.groupby(["product", "source"])
    .agg(
        n_events=("company_id", "size"),
        n_companies=("company_id", "nunique"),
        n_clean=("clean", "sum"),
        first_month_share=("pre_months", lambda s: float((s == 0).mean())),
        median_amount_eur=("amount_eur", "median"),
        first_event=("month", "min"),
        last_event=("month", "max"),
    )
    .reset_index()
    .sort_values("n_events", ascending=False)
)
events.to_csv(OUT / "A1_events.csv", index=False)
counts.to_csv(OUT / "A1_counts.csv", index=False)
print(f"{len(events)} eventos · {int(events['clean'].sum())} limpios · "
      f"{events['company_id'].nunique()} empresas con al menos un evento")
counts

**Acuerdo entre fuentes y eventos no anticipables.** La validación cruzada de A1: cuando una
empresa tiene primera cuota (`loan/installment`) y `created_at` de un préstamo, ¿caen a menos de
dos meses? Y la cuota de eventos que ocurren en el primer mes de historia de la empresa
(`pre_months == 0`), que no son anticipables por ningún modelo.

In [ ]:
first_by = (
    events.groupby(["product", "source", "company_id"])["month"].min().reset_index()
    .assign(mo=lambda d: pd.PeriodIndex(d["month"].astype(str), freq="M").asi8)
)


def agreement(product_a, source_a, product_b, source_b, tolerance=2):
    """Cuota de empresas con las dos trazas cuyo mes difiere en ≤ `tolerance` meses."""
    a = first_by.query("product == @product_a and source == @source_a").set_index("company_id")["mo"]
    b = first_by.query("product == @product_b and source == @source_b").set_index("company_id")["mo"]
    both = a.index.intersection(b.index)
    if not len(both):
        return 0, float("nan"), float("nan")
    delta = (a[both] - b[both]).to_numpy()
    return len(both), float((np.abs(delta) <= tolerance).mean()), float(np.median(delta))


n_both, share_agree, median_delta = agreement("loan", "installment", "loan", "created_at")
first_month_share = float((events["pre_months"] == 0).mean())
print(f"empresas con primera cuota y created_at de préstamo: {n_both}")
print(f"  |Δmeses| ≤ 2: {share_agree:.1%} · mediana Δ (cuota − created_at): {median_delta:+.0f} meses")
print(f"eventos en el primer mes de historia (pre_months == 0): {first_month_share:.1%}")
print(counts.set_index(["product", "source"])["first_month_share"].map("{:.1%}".format).to_string())

**Lectura de A1.** 1.682 eventos en 645 empresas, 324 limpios. El criterio de éxito de A1
(«≥ 50 eventos limpios en préstamo, anticipo/factoring y línea») **solo lo cumple el préstamo**:
78 por primera cuota y 67 por salto de cuota. Factoring (24), anticipo (28) y sobre todo línea
(11) se quedan muy por debajo, así que de esos productos A2 no puede decir gran cosa.

El acuerdo entre fuentes es flojo: de las 161 empresas con primera cuota y `created_at` de
préstamo, solo el 60 % caen a menos de dos meses, y la mediana del desfase es de −2 meses (la
cuota aparece **antes** que el `created_at`). Confirma lo de §2.2: `created_at` es fecha de
conexión a la plataforma, no de originación, y por eso es fuente secundaria.

El 23,8 % de los eventos ocurren en el primer mes de historia de la empresa, y en dos trazas eso
es la mitad o más (`line/line_tx` 57 %, `loan/installment` 42 %): son empresas que ya llegaron
con el producto contratado, no adopciones observadas. Ese es el motivo de que la ventana de
`clean_events` recorte tanto, y también de que `line/line_tx` se quede en 11 eventos.

## A2 — Estudio de eventos

Dos estimadores sobre el mismo panel y la misma tabla de eventos:

- **`matched_att`**: por cada evento, controles observados en t−1 en el mismo quintil de índice
  de estado dentro del mes, sin evento propio en [t−3, t+6]. El resultado es el cambio
  y(t+h) − y(t−1) (o el nivel y(t+h) si el resultado ya es binario). Trae la **pre-tendencia**,
  que es el contraste de falsación: si no es ~0, los dos grupos ya divergían antes.
- **`did_not_yet_treated`**: Callaway–Sant'Anna simplificado, cohortes por mes de evento y
  contrafactual las empresas que en g+h todavía no han adoptado.

Resultados: índice de estado, score, días de caja y DSCR como cambios; mes rojo (`n_red ≥ 2`) y
rotura de caja a 6 meses (`breach6`) como niveles. Horizontes 1, 3 y 6 meses. Los intervalos son
bootstrap; con n de 11 a 78 eventos son anchos por construcción.

Dos cosas que hay que tener delante al leer la tabla:

1. **`breach6` a t+6 pierde la mitad de los eventos.** Mirar la rotura en t+6 exige 12 meses de
   panel después del evento (6 del horizonte y 6 de la ventana de rotura), y `clean_events` solo
   exige 6. El n cae de 78 a 32 en `loan/installment` y de 11 a 4 en `line/line_tx`. El
   horizonte 1 es el análogo directo del «P(rotura en t+1…t+6)» del piloto de §2.5 y conserva
   el n del evento.
2. **`cash_buffer_days` y `dscr_6m` tienen colas enormes** (son ratios con denominador pequeño:
   saldo mínimo entre cargos diarios, servicio de deuda). Su ATT en media sale en decenas o
   cientos de miles y no es interpretable; se deja en la tabla por completitud. Lo que se lee
   de esos dos es el índice de estado, que ya los incorpora por rango dentro del mes.

In [ ]:
MATCHED_OUTCOMES = [
    ("state_index", False), ("score", False), ("cash_buffer_days", False), ("dscr_6m", False),
    ("red", True), ("breach6", True),
]
DID_OUTCOMES = [("state_index", False), ("breach6", True)]
A2_COLUMNS = ["product", "source", "estimator", "outcome", "h", "n_events", "att", "se",
              "ci_lo", "ci_hi", "pretrend_att", "pretrend_se"]

clean_by_pair = {}
rows = []
t_a2 = time.time()
for product, source in A2_PAIRS:
    clean = adoption.clean_events(events, product=product, source=source, min_pre=6, min_post=6)
    clean_by_pair[(product, source)] = clean
    if len(clean) < MIN_CLEAN_EVENTS:
        print(f"{product}/{source}: {len(clean)} eventos limpios < {MIN_CLEAN_EVENTS} → se salta")
        continue
    ev_keys = clean[["company_id", "month"]]
    for outcome, binary in MATCHED_OUTCOMES:
        out = eventstudy.matched_att(panel, ev_keys, outcome, horizons=HORIZONS, binary=binary)
        rows.append(out.assign(product=product, source=source, estimator="matched", outcome=outcome))
    for outcome, binary in DID_OUTCOMES:
        out = eventstudy.did_not_yet_treated(panel, ev_keys, outcome, horizons=HORIZONS, binary=binary)
        rows.append(out.assign(product=product, source=source, estimator="did", outcome=outcome,
                               pretrend_att=np.nan, pretrend_se=np.nan))
    n_coh = out["n_cohorts"].tolist()
    print(f"{product}/{source}: {len(clean)} eventos limpios · cohortes DiD {n_coh}")

att = pd.concat(rows, ignore_index=True)[A2_COLUMNS]
att.to_csv(OUT / "A2_att.csv", index=False)
print(f"\nA2_att.csv: {len(att)} filas en {time.time() - t_a2:.0f} s")
att.query("h == 6 and outcome == 'breach6'").round(4)

In [ ]:
att.query("h == 6 and estimator == 'matched' and outcome in ['state_index', 'cash_buffer_days', 'dscr_6m']").round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
for ax, outcome, title in zip(
    axes, ["breach6", "state_index"],
    ["P(rotura de caja en t+1…t+6)", "Índice de estado (cambio frente a t−1)"],
):
    sub = (
        att.query("estimator == 'matched' and h == 6 and outcome == @outcome")
        .assign(pair=lambda d: d["product"] + "/" + d["source"])
        .set_index("pair")
    )
    sub = sub.loc[[f"{p}/{s}" for p, s in A2_PAIRS if f"{p}/{s}" in sub.index]]
    x = np.arange(len(sub))
    lo = sub["att"] - sub["ci_lo"]
    hi = sub["ci_hi"] - sub["att"]
    ax.errorbar(x, sub["att"], yerr=[lo, hi], fmt="o", color="#1f4e79", capsize=4,
                markersize=7, label="ATT a t+6 (IC 95 % bootstrap)")
    ax.scatter(x, sub["pretrend_att"], facecolors="none", edgecolors="#c00000", s=70, zorder=3,
               label="pre-tendencia (falsación)")
    ax.axhline(0, color="grey", lw=1)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{i}\n(n={int(n)})" for i, n in zip(sub.index, sub["n_events"])],
                       rotation=45, ha="right", fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8, loc="best")
axes[0].set_ylabel("ATT (probabilidad; 0,10 = 10 puntos)")
axes[1].set_ylabel("ATT (unidades de índice)")
fig.suptitle("A2 · Qué pasa 6 meses después de adoptar, frente a controles emparejados", fontsize=12)
fig.tight_layout()
fig.savefig(OUT / "A2_att.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"figura en {OUT / 'A2_att.png'}")

**Lectura de A2.**

1. **El índice de estado baja después de adoptar cualquier cosa**: ATT a t+6 entre −0,03 y
   −0,08, con IC que excluye el cero en salto de cuota (−0,052), línea (−0,076), anticipo
   (−0,053) y las dos fuentes `created_at`. Es la mecánica que ya apuntaba el piloto: la cuota
   nueva entra en el DSCR, que pesa 0,20 en el índice. **No es deterioro, es contabilidad.**
2. **Sobre la rotura de caja el resultado es mixto y frágil.** La primera cuota de préstamo va
   con **más** rotura (a h = 1, +0,084 con IC 0,008–0,171 sobre 72 eventos; a h = 6, +0,100 con
   IC que roza el cero sobre 32), que es lo que se espera de la selección: quien empieza a pagar
   cuotas es quien acaba de necesitar dinero. El **salto de cuota** es el único efecto protector
   con n razonable (a h = 6, −0,111 con IC −0,202 a −0,023 sobre 34 eventos). La disposición
   apunta en la dirección mecánica esperada (−0,100 a h = 1) pero con IC que cruza el cero.
3. **Dos pre-tendencias invalidan su propio ATT**: `anticipo/description` (+0,19 sobre la rotura
   a h = 6) y `loan/created_at` (−0,19). Ahí los dos grupos ya divergían antes del evento y el
   ATT no se lee. En el resto la pre-tendencia es pequeña frente al efecto.
4. **`line/line_tx` a h = 6 tiene n = 4** y un IC estrecho que no significa nada: cuatro
   eventos remuestreados con reemplazo. Está en la figura por coherencia, no como evidencia.
5. **Los dos estimadores coinciden en signo en 15 de las 16 celdas de t+6** (la excepción es
   `loan/created_at` sobre la rotura, donde ambos son ~0). Es el contraste que importa: el
   emparejado y el DiD con no-tratados-aún no dependen de los mismos supuestos. Sobre el índice
   el DiD da magnitudes algo menores, lo esperable al no condicionar por quintil.

El criterio de éxito de A2 («pre-tendencia ~0 **y** ATT sobre rotura distinto de cero en algún
producto») se cumple técnicamente por el salto de cuota, con un solo producto y n = 34. La
conclusión operativa sigue siendo la de §2.5: **el generador contiene la mecánica del producto,
no una estructura causal de comportamiento**, y el simulador B1 no tiene mucho más que aprender
de estos datos.

## A3 — Propensión (modelo de comportamiento)

`behavior.fit_propensity` aprende «¿contrata este producto en (t, t+3]?» con LightGBM y
GroupKFold por `group_id`. Es una **línea base de comportamiento**, no una política óptima: dice
qué hacen las empresas parecidas, no qué les conviene.

Se ajustan dos conjuntos de eventos por producto: la ventana limpia de A2 (≥ 6 meses antes y
después, que es lo que exige medir un efecto) y el conjunto filtrado solo por producto y fuente
(`min_pre = min_post = 0`). La ventana no hace falta para **predecir** una adopción y recorta
mucho los positivos, así que el segundo es el titular y el primero queda como sensibilidad.

In [ ]:
fits = {}
prop_rows = []
for product, source in A3_PAIRS:
    for label, (min_pre, min_post) in {"ventana 6/6": (6, 6), "sin ventana": (0, 0)}.items():
        ev_p = adoption.clean_events(events, product=product, source=source,
                                     min_pre=min_pre, min_post=min_post)
        fit = behavior.fit_propensity(panel, ev_p, product, groups, horizon=A3_HORIZON, seed=SEED)
        fits[(product, source, label)] = fit
        prop_rows.append({
            "product": product, "source": source, "event_set": label, "n_events": len(ev_p),
            "n_rows": fit.n_rows, "n_pos": fit.n_pos, "auc_mean": fit.auc_mean,
            "auc_std": fit.auc_std, "oof_missing": float(fit.oof.isna().mean()),
            "top_importances": ", ".join(fit.importance.head(8).index),
        })

propensity = pd.DataFrame(prop_rows)
propensity.to_csv(OUT / "A3_propensity.csv", index=False)
propensity.drop(columns="top_importances").round(3)

In [ ]:
for r in prop_rows:
    if r["event_set"] == "sin ventana":
        print(f"{r['product']}/{r['source']}: AUC {r['auc_mean']:.3f} ± {r['auc_std']:.3f} "
              f"({r['n_pos']} positivos)\n  top-8: {r['top_importances']}")

**Calibración y solape.** La calibración compara la probabilidad media predicha fuera de fold
con la tasa observada por decil; el solape (histograma de `oof` para y = 1 y para y = 0) dice si
hay soporte común, que es lo que hace comparables tratados y controles en A2 y A4.

In [ ]:
fig, axes = plt.subplots(2, len(A3_PAIRS), figsize=(13, 7))
calib_tables = {}
for col, (product, source) in enumerate(A3_PAIRS):
    fit = fits[(product, source, "sin ventana")]
    target = behavior.adoption_target(
        panel, adoption.clean_events(events, product=product, source=source, min_pre=0, min_post=0),
        product, A3_HORIZON,
    )
    d = pd.DataFrame({"p": fit.oof.to_numpy(), "y": target["y"].to_numpy()}).dropna()
    bins = np.linspace(0, float(d["p"].max()), 31)
    top = axes[0, col]
    top.hist(d.loc[d["y"] == 0, "p"], bins=bins, density=True, alpha=0.6, label="y = 0", color="#9ecae1")
    top.hist(d.loc[d["y"] == 1, "p"], bins=bins, density=True, alpha=0.6, label="y = 1", color="#c00000")
    top.set_yscale("log")  # las dos clases se apilan cerca de cero: en lineal no se ve el solape
    top.set_title(f"{product}/{source} · AUC {fit.auc_mean:.2f}", fontsize=10)
    top.set_xlabel("propensión fuera de fold")
    top.set_ylabel("densidad (escala log)")
    top.legend(fontsize=8)

    d["bin"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False) + 1
    cal = d.groupby("bin").agg(n=("y", "size"), predicted=("p", "mean"), observed=("y", "mean"))
    calib_tables[f"{product}/{source}"] = cal
    bottom = axes[1, col]
    bottom.plot(cal["predicted"], cal["observed"], "o-", color="#1f4e79")
    lim = max(cal["predicted"].max(), cal["observed"].max()) * 1.1
    bottom.plot([0, lim], [0, lim], color="grey", lw=1, ls="--")
    bottom.set_xlabel("predicho (media del decil)")
    bottom.set_ylabel("observado")
fig.suptitle("A3 · Propensión a contratar en (t, t+3]: solape (arriba) y calibración (abajo)", fontsize=12)
fig.tight_layout()
fig.savefig(OUT / "A3_overlap.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"figura en {OUT / 'A3_overlap.png'}")
for name, cal in calib_tables.items():
    print(f"\n{name}\n{cal.round(4).to_string()}")

**Lectura de A3.** Con el conjunto sin ventana se reproduce el piloto de §2.4 casi clavado:
AUC 0,63 en primera cuota (664 positivos), 0,71 en primer uso de línea (117) y 0,73 en primer
factoring (252). **El criterio de éxito (AUC ≥ 0,65) lo pasan línea y factoring; el préstamo se
queda en 0,63.**

La ventana 6/6 no sirve para esto y los números lo enseñan: la línea cae a AUC 0,52 con 33
positivos y el 40 % de las filas sin probabilidad fuera de fold (folds de GroupKFold sin ningún
positivo). El préstamo sube a 0,76, que no es una mejora sino el artefacto contrario: exigir
6 meses de historia previa deja fuera precisamente las adopciones del primer mes (el 42 % de la
traza), que son las menos predecibles.

`months_of_history` está entre las tres primeras variables de los tres modelos, y en línea y
factoring aparecen además los saldos en euros: parte de lo que el modelo aprende es **la edad
de la conexión**, no el comportamiento (§2.4). El solape (histograma en escala log) cubre todo
el soporte de los positivos: cada fila con y = 1 tiene controles con propensión parecida, que es
la condición que necesitan A2 y A4. Al revés no pasa, y da igual: en línea y factoring hay
controles con propensión mucho más alta que cualquier tratado. La calibración, en cambio, está
sesgada: los deciles
bajos predicen menos de lo que ocurre y el decil alto predice más (0,196 frente a 0,111
observado en préstamo). **Sirve para ordenar, no para poner una probabilidad en la ficha.**

## A4 — Efectos heterogéneos (exploratorio)

X-learner con LightGBM sobre `breach6`: ¿el efecto de adoptar depende del colchón de caja?

- **Filas tratadas**: una por evento limpio, en su mes t, con las features de t−1.
- **Controles**: filas de empresas que nunca adoptan ese producto, en los mismos meses, con
  muestreo de hasta 5.000.
- **Split temporal**: eventos hasta 2025-08 entrenan, los posteriores son test.
- **Tratamientos**: `loan/installment`, y `disposicion/description ∪ line/line_tx` agrupados.

Dos configuraciones del X-learner. La **base** es la acordada (`min_child_samples=30`): con 12 a
33 filas tratadas en train, los modelos del brazo tratado **no pueden partir ni una vez**, así
que τ̂ sale casi constante y la heterogeneidad que se ve es la que filtra τ̂₀ pesada por la
propensión. La variante **leaf5** (`min_child_samples=5` solo en los modelos del brazo tratado y
de τ) es la única en la que la heterogeneidad puede expresarse; es ruidosísima y se lee como
exploratoria, nunca como una estimación. Todo A4 está dos órdenes de magnitud por debajo del
régimen de ~1.000 eventos que pide la literatura de uplift.

In [ ]:
lagged = panel.set_index(["company_id", "mo"])
FC = behavior.FEATURE_COLS


def design_matrix(treated_events, adopters, seed=SEED):
    """Filas (empresa, t): X en t−1, tratamiento w, resultado `breach6` en t."""
    months = sorted(treated_events["mo"].unique())
    grid = panel[panel["mo"].isin(months) & ~panel["company_id"].isin(adopters)]
    rows_d = pd.concat(
        [treated_events[["company_id", "mo"]].assign(w=1), grid[["company_id", "mo"]].assign(w=0)],
        ignore_index=True,
    )
    x = lagged[FC].reindex(pd.MultiIndex.from_arrays([rows_d["company_id"], rows_d["mo"] - 1]))
    y = lagged["breach6"].reindex(pd.MultiIndex.from_arrays([rows_d["company_id"], rows_d["mo"]]))
    out = rows_d.assign(**{c: x[c].to_numpy() for c in FC}, y=y.to_numpy())
    out = out[out["y"].notna() & out[FC].notna().any(axis=1)].reset_index(drop=True)
    control_idx = out.index[out["w"] == 0]
    if len(control_idx) > A4_CONTROLS:
        drop = np.random.default_rng(seed).choice(
            control_idx, len(control_idx) - A4_CONTROLS, replace=False
        )
        out = out.drop(index=drop).reset_index(drop=True)
    return out


def _probability(x, y, params, seed=SEED):
    """Clasificador LightGBM; si sólo hay una clase, la media constante."""
    if pd.Series(y).nunique() < 2:
        mean = float(np.mean(y))
        return lambda z: np.full(len(z), mean)
    model = LGBMClassifier(random_state=seed, **params).fit(x, y)
    return lambda z: model.predict_proba(z)[:, 1]


def xlearner(train, test, treated_params, seed=SEED):
    """τ̂(x) = g·τ̂₀ + (1−g)·τ̂₁ con g la propensión estimada (Künzel et al., 2019)."""
    treated, control = train[train["w"] == 1], train[train["w"] == 0]
    mu0 = _probability(control[FC], control["y"], A4_PARAMS, seed)
    mu1 = _probability(treated[FC], treated["y"], treated_params, seed)
    d1 = treated["y"].to_numpy() - mu0(treated[FC])
    d0 = mu1(control[FC]) - control["y"].to_numpy()
    tau1 = LGBMRegressor(random_state=seed, **treated_params).fit(treated[FC], d1)
    tau0 = LGBMRegressor(random_state=seed, **A4_PARAMS).fit(control[FC], d0)
    g = _probability(train[FC], train["w"], A4_PARAMS, seed)(test[FC])
    return g * tau0.predict(test[FC]) + (1 - g) * tau1.predict(test[FC])


def qini_curve(test, shares=np.arange(0.05, 1.001, 0.05)):
    """Curva estilo qini: resultado acumulado de tratados menos controles reescalados.

    Se ordena por τ̂ ascendente (primero la mayor reducción esperada de la rotura). Como el
    resultado es malo, un targeting que funciona deja la curva **por debajo** de la diagonal.
    """
    o = test.sort_values("tau", kind="stable")
    w, y = o["w"].to_numpy(), o["y"].to_numpy()
    n_t, n_c = np.cumsum(w), np.cumsum(1 - w)
    s_t, s_c = np.cumsum(w * y), np.cumsum((1 - w) * y)
    out = []
    for share in shares:
        k = max(int(round(share * len(o))), 1) - 1
        out.append({
            "bucket": round(float(share), 2), "n": k + 1, "n_treated": int(n_t[k]),
            "n_control": int(n_c[k]),
            "qini": s_t[k] - s_c[k] * n_t[k] / n_c[k] if n_c[k] else np.nan,
            "y_treated": s_t[k] / n_t[k] if n_t[k] else np.nan,
            "y_control": s_c[k] / n_c[k] if n_c[k] else np.nan,
        })
    curve = pd.DataFrame(out)
    curve["random"] = curve["qini"].iloc[-1] * curve["bucket"]
    return curve

In [ ]:
pooled = pd.concat([
    clean_by_pair[("disposicion", "description")], clean_by_pair[("line", "line_tx")]
])
TREATMENTS = {
    "loan/installment": (clean_by_pair[("loan", "installment")], {"loan"}, {"installment"}),
    "disposicion+line": (pooled, {"disposicion", "line"}, {"description", "line_tx"}),
}
CONFIGS = {"base": A4_PARAMS, "leaf5": dict(A4_PARAMS, min_child_samples=5)}
split_ordinal = pd.Period(A4_SPLIT, freq="M").ordinal

uplift_rows = []
monotonicity = {}
curves = {}
for name, (evs, products, sources) in TREATMENTS.items():
    first = evs.sort_values("month").drop_duplicates("company_id").copy()
    first["mo"] = pd.PeriodIndex(first["month"].astype(str), freq="M").asi8
    adopters = set(events.loc[events["product"].isin(products) & events["source"].isin(sources),
                              "company_id"])
    d = design_matrix(first, adopters)
    train, test = d[d["mo"] <= split_ordinal], d[d["mo"] > split_ordinal]
    print(f"\n=== {name}: {len(d)} filas ({int(d['w'].sum())} tratadas) · "
          f"train {len(train)} ({int(train['w'].sum())} tratadas) · "
          f"test {len(test)} ({int(test['w'].sum())} tratadas)")
    for config, params in CONFIGS.items():
        te = test.assign(tau=xlearner(train, test, params))
        quintile = pd.qcut(te["cash_buffer_days"].rank(method="first"), 5, labels=False) + 1
        table = (
            te.assign(bucket=quintile).groupby("bucket")
            .agg(n=("tau", "size"), n_treated=("w", "sum"),
                 cash_buffer_days=("cash_buffer_days", "median"),
                 tau_mean=("tau", "mean"), tau_std=("tau", "std"),
                 y_treated=("y", lambda s: float(s[te.loc[s.index, "w"] == 1].mean())),
                 y_control=("y", lambda s: float(s[te.loc[s.index, "w"] == 0].mean())))
            .reset_index()
        )
        table["n_control"] = table["n"] - table["n_treated"]
        rho = float(spearmanr(table["bucket"], table["tau_mean"]).statistic)
        monotone = bool(abs(rho) >= 0.9)
        monotonicity[f"{name}|{config}"] = {
            "spearman": round(rho, 3), "monotone": monotone,
            "tau_sd": round(float(te["tau"].std()), 5),
            "tau_mean": round(float(te["tau"].mean()), 5),
            "n_test_treated": int(te["w"].sum()),
        }
        curve = qini_curve(te)
        curves[f"{name}|{config}"] = curve
        uplift_rows.append(table.assign(treatment=name, config=config, block="quintile"))
        uplift_rows.append(curve.assign(treatment=name, config=config, block="qini"))
        print(f"  [{config}] τ̂ media {te['tau'].mean():+.4f} · sd {te['tau'].std():.5f} · "
              f"Spearman(quintil, τ̂) {rho:+.2f} → monótono: {monotone}")
        print(table.round(4).to_string(index=False))

uplift = pd.concat(uplift_rows, ignore_index=True)[
    ["treatment", "config", "block", "bucket", "n", "n_treated", "n_control",
     "cash_buffer_days", "tau_mean", "tau_std", "y_treated", "y_control", "qini", "random"]
]
uplift.to_csv(OUT / "A4_uplift.csv", index=False)
print(f"\nA4_uplift.csv: {len(uplift)} filas")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, _) in zip(axes, TREATMENTS.items()):
    for config, style in [("base", "--"), ("leaf5", "-")]:
        curve = curves[f"{name}|{config}"]
        ax.plot(curve["bucket"], curve["qini"], style, marker="o", markersize=4,
                label=f"τ̂ {config}")
    ax.plot(curve["bucket"], curve["random"], color="grey", lw=1, label="aleatorio")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("cuota de la cartera de test objetivo (ordenada por τ̂ ascendente)")
    ax.set_ylabel("roturas acumuladas: tratados − controles")
    ax.legend(fontsize=8)
fig.suptitle("A4 · Curva estilo qini sobre los eventos de test (exploratorio, n de 15 a 45)", fontsize=12)
fig.tight_layout()
fig.savefig(OUT / "A4_uplift.png", dpi=140, bbox_inches="tight")
plt.close(fig)
print(f"figura en {OUT / 'A4_uplift.png'}")

**Lectura de A4. Se archiva, como preveía el documento.**

- En la configuración **base** τ̂ sale prácticamente constante (sd 0,007 en préstamo y 0,00007
  en el grupo disposición+línea). No es un hallazgo sobre el mundo: con 33 y 12 filas tratadas
  en train, `min_child_samples=30` impide **cualquier** partición en los modelos del brazo
  tratado, así que τ̂₁ es una constante y la poca variación que queda es la de τ̂₀ pesada por una
  propensión ~0,01. Con esta configuración A4 no mide heterogeneidad; mide su propia falta de
  datos.
- En la variante **leaf5**, disposición+línea da el patrón exacto de la hipótesis: Spearman
  +1,00, con τ̂ de −0,31 en el quintil de menos colchón a +0,04 en el de más colchón (el dinero
  que entra en la cuenta vale cuando no hay caja, y no vale cuando la hay). Es bonito y **no es
  creíble**: sale de 12 eventos de entrenamiento y 15 de test, con τ̂ de desviación típica 0,28
  sobre un resultado cuya base es 0,18. El préstamo, en la misma variante, da Spearman −0,60 y
  τ̂ medio +0,12, coherente con el ATT positivo de A2 pero igual de inestable.
- Las curvas qini hay que leerlas con la misma advertencia. La de disposición+línea en base
  queda por debajo del aleatorio en tres cuartos de la cartera, que parecería un targeting que
  funciona; pero con τ̂ constante hasta la quinta cifra **el orden lo decide el desempate**
  (orden original de las filas), no el modelo. La curva informativa es la de leaf5, y ahí el
  escalón depende de dónde caigan cuatro o cinco eventos.

Conclusión: la bandera de monotonía es `true` en un solo caso y solo en la variante que puede
partir. A4 está dos órdenes de magnitud por debajo del régimen de ~1.000 eventos que pide la
literatura de uplift; **no entra en ninguna pantalla ni en ninguna recomendación**.

## Resumen

`A_summary.json` reúne lo que consumen el informe y la ficha de transparencia: eventos por
producto y fuente (brutos y limpios), ATT a 6 meses sobre rotura e índice con su intervalo y su
pre-tendencia, AUC de propensión y banderas de monotonía del uplift.

In [ ]:
def att_entry(product, source, outcome):
    row = att.query(
        "product == @product and source == @source and estimator == 'matched' "
        "and outcome == @outcome and h == 6"
    )
    if row.empty:
        return None
    r = row.iloc[0]
    out = {k: (None if pd.isna(r[k]) else round(float(r[k]), 4))
           for k in ["att", "se", "ci_lo", "ci_hi", "pretrend_att", "pretrend_se"]}
    return {"n_events": int(r["n_events"]), **out}


summary = {
    "generated_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "panel": {
        "rows": int(len(panel)), "companies": int(panel["company_id"].nunique()),
        "first_month": str(panel["month"].min()), "last_month": str(panel["month"].max()),
        "breach6_rate": round(float(panel["breach6"].mean()), 4),
    },
    "a1": {
        "n_events_total": int(len(events)), "n_clean_total": int(events["clean"].sum()),
        "by_pair": {
            f"{r.product}/{r.source}": {"raw": int(r.n_events), "clean": int(r.n_clean),
                                        "companies": int(r.n_companies),
                                        "first_month_share": round(float(r.first_month_share), 4)}
            for r in counts.itertuples()
        },
        "installment_vs_created_at": {
            "n_companies_both": int(n_both), "share_within_2_months": round(share_agree, 4),
            "median_delta_months": median_delta,
        },
        "first_month_share_all": round(first_month_share, 4),
    },
    "a2": {
        f"{p}/{s}": {"breach6_h6": att_entry(p, s, "breach6"),
                     "state_index_h6": att_entry(p, s, "state_index")}
        for p, s in A2_PAIRS
    },
    "a3": {
        f"{r['product']}/{r['source']}|{r['event_set']}": {
            "n_events": int(r["n_events"]), "n_pos": int(r["n_pos"]),
            "auc_mean": None if pd.isna(r["auc_mean"]) else round(float(r["auc_mean"]), 4),
            "auc_std": None if pd.isna(r["auc_std"]) else round(float(r["auc_std"]), 4),
        }
        for r in prop_rows
    },
    "a4": monotonicity,
    "wall_time_s": round(time.time() - T0, 1),
}
(OUT / "A_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
print("ficheros en", OUT)
for path in sorted(OUT.glob("A*")):
    print(f"  {path.name:22s} {path.stat().st_size / 1024:8.1f} KB")
print(f"\ntiempo total {summary['wall_time_s']} s")

## Qué se lleva el equipo de la pista A

1. **A1 solo cumple su criterio en el préstamo** (78 eventos limpios por primera cuota, 67 por
   salto de cuota). Línea (11), factoring (24), anticipo (28) y disposición (17) se quedan
   lejos de los 50 que pedía el documento, sobre todo porque una parte grande de las trazas
   aparece en el primer mes de historia de la empresa (57 % en línea, 42 % en préstamo): ya
   venían con el producto.
2. **A2 confirma el piloto: el generador tiene mecánica de producto, no comportamiento.** El
   índice de estado baja tras cualquier adopción (contabilidad del DSCR, no deterioro) y sobre
   la rotura de caja solo el salto de cuota se separa de cero con n razonable (−0,111, IC
   −0,202 a −0,023, n = 34). La primera cuota va con **más** rotura (+0,084 a h = 1, IC
   0,008–0,171), que es selección. Para el pitch, esto es lo que hay que decir en la ficha de
   transparencia, con la figura `A2_att.png` al lado.
3. **A3 vale como línea base de comportamiento** («el X % de las empresas en tu situación abrió
   una línea en los tres meses siguientes»), con AUC 0,71 en línea y 0,73 en factoring, pero
   **no como probabilidad calibrada**: la calibración está sesgada y parte de la señal es la
   edad de la conexión.
4. **A4 se archiva.** Con 12–33 eventos de entrenamiento no hay heterogeneidad que estimar.
5. Para B1, la consecuencia práctica: la validación cruzada «el simulador tiene que reproducir
   A2» solo tiene músculo estadístico en dos celdas (primera cuota y salto de cuota). En el
   resto de productos el intervalo de A2 es tan ancho que casi cualquier simulador pasaría la
   prueba, y hay que decirlo al presentarla.